In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

/Users/muhammadnehal/Desktop/mindlens/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("/Users/muhammadnehal/Desktop/mindlens/datasets/text/mindlens_text_dataset.csv")

In [4]:
df.head(50)

,text,target
0,I'm almost at a breaking point...New Year's Ev...,1
1,There’s no joy leftI spent too much of my life...,1
2,I often cut myself. But of sanity not sadnessI...,1
3,Please just someone talk to meI'm crying and o...,1
4,Little do my friends know that they are the on...,0
5,so dang tiredI have a neurological disorder th...,1
6,My social anxiety is killing me.Sorry for the ...,1
7,Holy shit is today awfulI am having troubles l...,1
8,"We girls watch porn too, it's not just a boy t...",0
9,ask a girl some questions I'm really bored and...,0


In [5]:
df.shape

(239805, 2)

In [6]:
df['target'].value_counts()

target
0    119937
1    119868
Name: count, dtype: int64

TRAIN/VALIDATION SPLIT or TRAIN/TEST SPLIT

In [6]:
train_texts, val_texts, train_targets, val_targets = train_test_split(
    df["text"],
    df["target"],
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)

CONVERTING HUFFINGFACE DATASET

In [7]:
train_dataset = Dataset.from_dict({
    "text": train_texts.tolist(),
    "target": train_targets.tolist()
})

val_dataset = Dataset.from_dict({
    "text": val_texts.tolist(),
    "target": val_targets.tolist()
})

In [8]:
print(train_dataset)

Dataset({
    features: ['text', 'target'],
    num_rows: 191844
})


In [9]:
print(val_dataset)

Dataset({
    features: ['text', 'target'],
    num_rows: 47961
})


LOAD ROBERTa TOKENIZER

In [10]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

In [11]:
from huggingface_hub import login

In [12]:
login('hf_YOUR_TOKEN_HERE')

In [13]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

TOKENIZATION FUNCTION

In [14]:
def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

APPLYING TOKENIZER

In [15]:
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 47961/47961 [00:03<00:00, 14350.25 examples/s]


FORMATTING DATASET FOR Pytorch

In [16]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","target"], # Temporarily keep 'target' here
    # The below line specifies how to map the 'target' column to 'labels'
    rename_columns={'target': 'labels'}
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","target"], # Temporarily keep 'target' here
    # The below line specifies how to map the 'target' column to 'labels'
    rename_columns={'target': 'labels'}
)

LOADING PRETRAINED ROBERTa MODEL

In [17]:
num_labels = len(df["target"].unique())

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=num_labels
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 13857.68it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DEFINE EVALUATION METRICS


In [18]:
def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

TRAINING CONFIGURATION

In [20]:
import os
from transformers import TrainingArguments

os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    #  Safe for Mac
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    num_train_epochs=2,
    weight_decay=0.01,

    report_to="tensorboard",
    load_best_model_at_end=True,
    logging_steps=20,

    #  Important for stability
    dataloader_num_workers=0,

    # simulate larger batch
    gradient_accumulation_steps=4,
)

INITIALIZE TRAINER

In [21]:
# Rename 'target' to 'labels' for the trainer to recognize it
train_dataset = train_dataset.rename_column("target", "labels")
val_dataset = val_dataset.rename_column("target", "labels")

In [23]:
# 1. Completely clear any previous formatting/state

train_dataset.reset_format()
val_dataset.reset_format()

# 2. Re-apply the rename ONLY if the column is still called 'target'

if "target" in train_dataset.column_names:
    train_dataset = train_dataset.rename_column("target", "labels")
if "target" in val_dataset.column_names:
    val_dataset = val_dataset.rename_column("target", "labels")

# 3. Manually set the format to 'torch' and specify the columns
# This forces the dataset to drop any 'ghost' arguments like 'rename_columns'

columns = ["input_ids", "attention_mask", "labels"]
train_dataset.set_format(type="torch", columns=columns)
val_dataset.set_format(type="torch", columns=columns)

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,  # Fixed: changed from 'tokenizer'
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
import pickle

# Define your file path
model_path = "text_model.pkl"

# Save the model
with open(model_path, "wb") as f:
    pickle.dump(trainer.model, f)

print(f"Model saved to {model_path}")